# CS345 Fall 2026 Final Project -- Amazon Review Sentiment Classifier

## Part 1 Preparing the Dataset

Abstract

In [23]:
import matplotlib.pyplot as plt
import numpy as np
import json
import pandas as pd

raw_text_data = pd.read_json(
    "Health_and_Personal_Care.jsonl.gz",
    lines=True,
    #nrows=5
)

raw_text_meta = pd.read_json(
    "meta_Health_and_Personal_Care.jsonl.gz",
    lines=True,
    #nrows=5
)

# Check data shape
print(raw_text_data.shape)
print(raw_text_meta.shape)

# print(raw_text_data["rating"].unique())

# Create sentiment map:
sentiment_map = {1: 0, 2: 0, 4: 1, 5: 1}

# Apply sentiment mask:
raw_text_data["sentiment"] = raw_text_data["rating"].map(sentiment_map)

# Drop rows with labels without 0 or 1:
labeled_reviews = raw_text_data.dropna(subset=["sentiment"]).copy()

print(labeled_reviews.shape)

#print(labeled_reviews[["rating", "sentiment"]].head(10))
#print(labeled_reviews["sentiment"].unique())

# Concatenate data sets to one set and check it's shape
combined_reviews = labeled_reviews.merge(
    # Drop the average_rating to avoid data pollution.
    raw_text_meta.drop(columns=["average_rating"]),
    # Match reviews to products.
    on="parent_asin",
    # Preserve all the reviews including without metadata
    how="left",
    # Distinguish overlaps 
    suffixes=("_review", "_product"),
    # One product ID per metadata record to avoid dupes
    validate="many_to_one"
)

print(combined_reviews[
    ["parent_asin", "title_review", "title_product"] 
].head())



(494121, 10)
(60293, 14)
(457172, 11)
  parent_asin                                       title_review  \
0  B07TDSJZMR  12 mg is 12 on the periodic table people! Mg f...   
1  B08637FWWF                 Save the lanet using less plastic.   
2  B07KJVGNN5                                          Fantastic   
3  B092RP73CX  It holds the water and makes bubbles.  That's ...   
4  B08KYJLF5T                                         Not for me   

                                       title_product  
0  High Potency Magnesium Citrate Capsules 1000mg...  
1  Evolve Concentrated Bleach Tablets - 32-ct Bot...  
2  Dr. Foot's Gel Heel Protectors, Plantar Fascii...  
3  Homedics Bubble Bliss Deluxe-Foot Spa, Heat Ma...  
4  Brain Supplement 1053mg - Premium Nootropic Br...  


In [24]:
# Concatenate data sets to one set and check it's shape
combined_reviews = labeled_reviews.merge(
    # Drop the average_rating to avoid data pollution.
    raw_text_meta.drop(columns=["average_rating"]),
    # Match reviews to products.
    on="parent_asin",
    # Preserve all the reviews including without metadata
    how="left",
    # Distinguish overlaps 
    suffixes=("_review", "_product"),
    # One product ID per metadata record to avoid dupes
    validate="many_to_one"
)

print(combined_reviews[
    ["parent_asin", "title_review", "title_product"] 
].head())

  parent_asin                                       title_review  \
0  B07TDSJZMR  12 mg is 12 on the periodic table people! Mg f...   
1  B08637FWWF                 Save the lanet using less plastic.   
2  B07KJVGNN5                                          Fantastic   
3  B092RP73CX  It holds the water and makes bubbles.  That's ...   
4  B08KYJLF5T                                         Not for me   

                                       title_product  
0  High Potency Magnesium Citrate Capsules 1000mg...  
1  Evolve Concentrated Bleach Tablets - 32-ct Bot...  
2  Dr. Foot's Gel Heel Protectors, Plantar Fascii...  
3  Homedics Bubble Bliss Deluxe-Foot Spa, Heat Ma...  
4  Brain Supplement 1053mg - Premium Nootropic Br...  


In [25]:
# Create test/train splits
y = combined_reviews["sentiment"]

X = combined_reviews.drop(columns=["rating", "sentiment"])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [26]:
# Helper method for converting dicts and lists  to values:

def value_to_text(value):
    if isinstance(value, dict):
        return " ".join(
            f"{key} {value_to_text(item)}"
            for key, item in value.items()
        )

    if isinstance(value, list):
        return " ".join(value_to_text(item) for item in value)

    if pd.isna(value):
        return ""

    return str(value)

In [27]:
# Helper method for combining text columns into one:
def combine_text(frame):
    text_columns = [
        "title_review",
        "text",
        "title_product",
        "features",
        "description",
        "categories",
        "details"
    ]

    text_values = frame[text_columns].apply(
        lambda column: column.map(value_to_text)
    )

    return text_values.agg(" ".join, axis=1)

In [29]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline

text_pipe = make_pipeline(
    FunctionTransformer(combine_text, validate=False),
    TfidfVectorizer()
)

In [30]:
# Translate purchase verification booleans into [0, 1]

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["price", "rating_number", "verified_purchase"]
num_pipe = make_pipeline(
    # Replace missing values with the column's training median, flag missing values
    SimpleImputer(strategy="median", add_indicator=True),
    # Standardize the features using training means and std devs.
    StandardScaler()
)

In [31]:
# Add categories:

category_columns = ["store", "main_category"]

# replace None and NaN with the same label--"__missing__"
def clean_categories(frame):
    return frame.fillna("__missing__").astype(str)

# Create indicator columns for categories.
categorical_pipe = make_pipeline(
    FunctionTransformer(clean_categories, validate=False),
    # allow validation reviews to have stores absent from the training fold - they'll be "0".
    OneHotEncoder(handle_unknown="ignore")
)

In [32]:
# Helper for creating counts of the media lists and components by timestamp

def add_features(frame):
    result = frame.copy()

    for source, target in [
        ("images_review", "review_image_count"),
        ("images_product", "product_image_count"),
        ("videos", "product_video_count")
    ]:
        result[target] = result[source].map(
            lambda value: len(value) if isinstance(value, list) else np.nan
        )

    timestamps = result["timestamp"]

    if pd.api.types.is_numeric_dtype(timestamps):
        dates = pd.to_datetime(
            timestamps, unit="ms", errors="coerce", utc=True
        )
    else:
        # Invalid dates are treated as missing
        dates = pd.to_datetime(
            timestamps, errors="coerce", utc=True
        )

    result["review_year"] = dates.dt.year
    result["review_month"] = dates.dt.month.astype("Int64").astype("string")
    result["review_weekday"] = dates.dt.day_of_week.astype("Int64").astype("string")

    return result

In [33]:
# Make column lists by feature types:

text_cols = [
    "title_review", "text", "title_product",
    "features", "description", "categories", "details"
]

num_cols = [
    "price", "rating_number", "verified_purchase",
    "review_image_count", "product_image_count",
    "product_video_count", "review_year"
]

category_columns = [
    "store", "main_category", "review_month", "review_weekday"
]

In [37]:
# Combine the lists using ColumnTransformer

from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline

svc = LinearSVC(C=1.0)
baseline = svc

preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_pipe, text_cols),
        ("numeric", num_pipe, num_cols),
        ("categorical", categorical_pipe, category_columns)
    ],
    remainder="drop",
    sparse_threshold=1.0
)

baseline = make_pipeline(
    FunctionTransformer(add_features, validate=False),
    preprocessor,
    svc
)

In [38]:
from sklearn.metrics import classification_report

X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

# Learn preprocessing settings and train the classifier.
baseline.fit(X_fit, y_fit)

# Predict labels for reviews withheld from this fit.
baseline_predictions = baseline.predict(X_valid)

# Inspect a few predictions alongside their actual labels.
preview = pd.DataFrame({
    "actual": y_valid,
    "predicted": baseline_predictions
})

print(preview.head(10))

# Print a full classification report for the baseline.
print(classification_report(
    y_valid,
    baseline_predictions,
    labels=[0, 1],
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0
))

        actual  predicted
7864       0.0        1.0
388845     1.0        1.0
167274     1.0        1.0
193153     0.0        0.0
389353     0.0        0.0
233254     1.0        1.0
223252     1.0        1.0
253516     1.0        1.0
145604     1.0        1.0
406435     1.0        1.0
              precision    recall  f1-score   support

    negative     0.8827    0.8027    0.8408     15754
    positive     0.9472    0.9707    0.9588     57394

    accuracy                         0.9345     73148
   macro avg     0.9149    0.8867    0.8998     73148
weighted avg     0.9333    0.9345    0.9334     73148



In [36]:
##@article{hou2024bridging,
  ##title={Bridging Language and Items for Retrieval and Recommendation},
  ##author={Hou, Yupeng and Li, Jiacheng and He, Zhankui and Yan, An and Chen, Xiusi and McAuley, Julian},
  ##journal={arXiv preprint arXiv:2403.03952},
  ##year={2024}
##}